In [1]:
!pip install --default-timeout=100 kaleido rt-utils

In [6]:
!pip install scikit-image

   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
    --------------------------------------- 0.3/12.8 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/12.8 MB 3.9 MB/s eta 0:00:03
   ---- ----------------------------------- 1.6/12.8 MB 4.4 MB/s eta 0:00:03
   ------ --------------------------------- 2.1/12.8 MB 2.6 MB/s eta 0:00:05
   ------ --------------------------------- 2.1/12.8 MB 2.6 MB/s eta 0:00:05
   ---------- ----------------------------- 3.4/12.8 MB 2.8 MB/s eta 0:00:04
   ---------- ----------------------------- 3.4/12.8 MB 2.8 MB/s eta 0:00:04
   ---------- ----------------------------- 3.4/

In [8]:
import pandas as pd
import os

# Define paths
ahsania_path = '../Results/amcgh_dosiomics.csv'
square_path = '../Results/square_dosiomics.csv'
output_path = '../Results/final_local_dosiomics.csv'

print("Loading datasets...")

# Load Ahsania
try:
    df_ahsania = pd.read_csv(ahsania_path)
    print(f"Ahsania Patients: {len(df_ahsania)}")
except:
    df_ahsania = pd.DataFrame()
    print("Warning: Ahsania file not found.")

# Load Square
try:
    df_square = pd.read_csv(square_path)
    print(f"Square Patients: {len(df_square)}")
except:
    df_square = pd.DataFrame()
    print("Warning: Square file not found.")

# Merge into df_final
df_final = pd.concat([df_ahsania, df_square], ignore_index=True)

if not df_final.empty:
    print(f"SUCCESS: Merged {len(df_final)} patients into df_final.")
else:
    print("Error: No data loaded.")

Loading datasets...
Ahsania Patients: 50
Square Patients: 22
SUCCESS: Merged 72 patients into df_final.


In [9]:
import plotly.express as px
import plotly.io as pio

# Set default template
pio.templates.default = "plotly_white"

if not df_final.empty:
    print("Generating Interactive Plotly Graph...")

    # Create Box Plot with underlying points (Jitter)
    fig = px.box(
        df_final, 
        x='Source', 
        y='D95_Gy', 
        color='Source',
        points='all',  # Shows all individual patient dots
        hover_data=['PatientID'], # Hover over a dot to see the Patient ID!
        color_discrete_map={'Ahsania': '#EF553B', 'Square': '#636EF2'}, # Modern Red/Blue
        title='<b>Tumor Coverage Comparison</b> (Target D95)',
        labels={'D95_Gy': 'Dose (Gy) covering 95% of Target', 'Source': 'Hospital'}
    )

    # Polish the layout
    fig.update_layout(
        width=800,
        height=600,
        showlegend=False,
        font=dict(size=14),
        yaxis=dict(showgrid=True, gridcolor='#eee'),
        title_x=0.5 # Center title
    )

    # Show the plot
    fig.show()

    # Save as HTML (interactive)
    fig.write_html("../Results/Figure_Dosiomics_Interactive.html")
    print("Saved to ../Results/Figure_Dosiomics_Interactive.html")
    
    # Try saving PNG
    try:
        fig.write_image("../Results/Figure_Dosiomics_Comparison.png", scale=3)
        print("Static image saved.")
    except:
        print("Note: To save PNG, install kaleido: pip install kaleido")
else:
    print("Dataframe is empty.")

Generating Interactive Plotly Graph...


Saved to ../Results/Figure_Dosiomics_Interactive.html
Static image saved.


In [11]:
import os
import numpy as np
import pandas as pd
import pydicom
import SimpleITK as sitk
from dicompylercore import dicomparser
from radiomics import featureextractor
from skimage.draw import polygon
import logging
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")
logger = logging.getLogger("radiomics")
logger.setLevel(logging.ERROR)

# --- CONFIGURATION ---
data_sources = {
    'Ahsania': '../Data/AMCGH/',      
    'Square': '../Data/SQUARE/'
}

output_csv = '../Results/local_radiomics_multicenter.csv'
os.makedirs('../Results', exist_ok=True)

# Setup Radiomics Extractor
params = {}
extractor = featureextractor.RadiomicsFeatureExtractor(**params)
extractor.settings['binWidth'] = 25
extractor.settings['resampledPixelSpacing'] = [1, 1, 1]
extractor.settings['interpolator'] = sitk.sitkBSpline
extractor.enableAllImageTypes()

print("Radiomics Pipeline Initialized (Robust Mode).")

# --- HELPER 1: Find Tumor ROI (Fixed) ---
def find_target_roi_radiomics(structure_dict):
    # Priority List: GTV > CTV > PTV
    target_priorities = [
        'gtv', 'gtv_t', 'gtv t', 'gtv_total', 'gtv total', 
        'ctv', 'ctv_t', 'ctv t', 
        'ptv', 'ptv_total', 'ptv total', 'ptv_plan'
    ]
    
    # FIX: Correctly access the 'name' field inside the dictionary
    clean_to_id = {}
    for key, val in structure_dict.items():
        if 'name' in val:
            name = val['name']
            clean = name.lower().strip().replace(' ','').replace('_','').replace('-','')
            clean_to_id[clean] = key

    # Check matches
    for priority in target_priorities:
        clean_p = priority.replace(' ','').replace('_','').replace('-','')
        for clean_name, roi_id in clean_to_id.items():
            if clean_p in clean_name:
                # Return ID and Original Name
                return roi_id, structure_dict[roi_id]['name']
                
    return None, None

# --- HELPER 2: Manual Mask Generation ---
def create_mask_from_contour(dicom_paths, rtstruct_path, roi_id):
    # Load CT Image Series
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames(dicom_paths)
    try:
        image_sitk = reader.Execute()
    except:
        return None, None
    
    # Get image array and metadata
    image_arr = sitk.GetArrayFromImage(image_sitk)
    
    # Load RTStruct
    rtss = dicomparser.DicomParser(rtstruct_path)
    try:
        contours = rtss.GetStructureCoordinates(roi_id)
    except:
        return None, None
    
    # Create empty mask
    mask_arr = np.zeros_like(image_arr, dtype=np.uint8)
    
    if not contours: return None, None

    # Map Z-coordinates to slice indices
    z_slices = [image_sitk.TransformIndexToPhysicalPoint([0,0,i])[2] for i in range(image_sitk.GetDepth())]
    
    for z_pos, points in contours.items():
        # Find closest slice index
        z_idx = np.argmin(np.abs(np.array(z_slices) - float(z_pos)))
        
        pixel_points = []
        for p in points:
            # Transform physical point -> index
            idx = image_sitk.TransformPhysicalPointToIndex([p[0], p[1], float(z_pos)])
            pixel_points.append([idx[1], idx[0]]) # (row, col)
            
        pixel_points = np.array(pixel_points)
        if len(pixel_points) > 0:
            rr, cc = polygon(pixel_points[:, 0], pixel_points[:, 1], shape=(image_arr.shape[1], image_arr.shape[2]))
            mask_arr[z_idx, rr, cc] = 1
        
    mask_sitk = sitk.GetImageFromArray(mask_arr)
    mask_sitk.CopyInformation(image_sitk)
    
    return image_sitk, mask_sitk

# --- MAIN LOOP ---
results = []

for source_name, source_path in data_sources.items():
    if not os.path.exists(source_path): continue
    
    patient_folders = sorted([f for f in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, f))])
    print(f"\nProcessing {len(patient_folders)} patients in {source_name}...")
    
    for i, patient_id in enumerate(patient_folders):
        patient_dir = os.path.join(source_path, patient_id)
        
        ct_files = []
        rt_struct_path = None
        
        # 1. ROBUST FILE SEARCH (Using pydicom to be 100% sure)
        for root, dirs, files in os.walk(patient_dir):
            for f in files:
                full_path = os.path.join(root, f)
                try:
                    # Force read allows us to read Ahsania files without .dcm extension or standard headers
                    dcm = pydicom.dcmread(full_path, stop_before_pixels=True, force=True)
                    mod = dcm.get("Modality", "Unknown")
                    
                    if mod == 'CT':
                        ct_files.append(full_path)
                    elif mod == 'RTSTRUCT':
                        rt_struct_path = full_path
                except:
                    continue
        
        # Sort CT files by Instance Number to ensure correct 3D volume construction
        ct_files.sort()

        if not ct_files or not rt_struct_path:
            print(f"[{i+1}] {patient_id}: [SKIP] Missing Files (CT: {len(ct_files)}, Struct: {rt_struct_path is not None})")
            continue

        try:
            # 2. Parse Struct
            rtss = dicomparser.DicomParser(rt_struct_path)
            structures = rtss.GetStructures()
            
            # 3. Find Tumor
            roi_id, roi_name = find_target_roi_radiomics(structures)
            
            if roi_id:
                # 4. Create Mask & Extract
                image, mask = create_mask_from_contour(ct_files, rt_struct_path, roi_id)
                
                if image and mask:
                    features = extractor.execute(image, mask)
                    
                    row = {'PatientID': patient_id, 'Source': source_name, 'ROI_Name': roi_name}
                    for k, v in features.items():
                        if 'diagnostics' not in k: row[k] = v
                    results.append(row)
                    print(f"[{i+1}] {patient_id}: Success ({roi_name})")
                else:
                    print(f"[{i+1}] {patient_id}: [FAIL] Mask Generation")
            else:
                avail = [s['name'] for k,s in structures.items()]
                print(f"[{i+1}] {patient_id}: [SKIP] No Tumor. Avail: {avail[:3]}")
                
        except Exception as e:
            print(f"[{i+1}] {patient_id}: [ERROR] {str(e)}")

# Save
if results:
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"\nSUCCESS! Saved {len(df)} patients to {output_csv}")
else:
    print("\nNo data extracted.")

Radiomics Pipeline Initialized (Robust Mode).

Processing 52 patients in Ahsania...
[1] 1042new: [ERROR] 0
[2] 1083New: [ERROR] 0
[3] 1102 new: [ERROR] 0
[4] 1115new: [ERROR] 0
[5] 1127new: [ERROR] 0
[6] 1161new: [ERROR] 0
[7] 1209new: [ERROR] 0
[8] 1233new: [ERROR] 0
[9] 1299new: [ERROR] 0
[10] 1342new: [ERROR] 0
[11] 1356new: [ERROR] 0
[12] 1392new: [ERROR] 0
[13] 1430new: [ERROR] 0
[14] 1495new: [ERROR] 0
[15] 20230367new: [ERROR] 0
[16] 20230396(39): [ERROR] 0
[17] 20230490new: [ERROR] 0
[18] 20250013: [ERROR] 0
[19] 20250106: [ERROR] 0
[20] 20250117: [ERROR] 0
[21] 20250146: [ERROR] 0
[22] 20250154: [ERROR] 0
[23] 20250982new: [ERROR] 0
[24] 20251575new: [ERROR] 0
[25] 20251577new: [ERROR] 0
[26] 20251626new: [ERROR] 0
[27] 20251673new: [ERROR] 0
[28] 20251705new: [ERROR] 0
[29] 20251727new: [ERROR] 0
[30] 20251773: [ERROR] 0
[31] 832: [ERROR] 0
[32] 956new: [ERROR] 0
[33] 995new: [ERROR] 0
[34] New 20250871: [ERROR] 0
[35] New 20251222: [ERROR] 0
[36] New 20251320: [ERROR] 0
[37]